In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
from settings import IRIS
df = pd.read_csv(IRIS)
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [12]:
y = df['Species']
X = df.drop(columns=['Species' , 'Id'])
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2,stratify=y)
print(X_train.shape , X_test.shape , y_train.shape , y_test.shape)

(120, 4) (30, 4) (120,) (30,)


In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
y_train

143     Iris-virginica
123     Iris-virginica
85     Iris-versicolor
57     Iris-versicolor
7          Iris-setosa
            ...       
38         Iris-setosa
113     Iris-virginica
136     Iris-virginica
124     Iris-virginica
126     Iris-virginica
Name: Species, Length: 120, dtype: str

### Soft Margin SVM Objective Function

The Soft Margin SVM minimizes the following loss function:

$$L(w, b) = \frac{1}{2} ||w||^2 + C \sum_{i=1}^{n} \max(0, 1 - y_i(w \cdot x_i + b))$$

Where:
* $\frac{1}{2} ||w||^2$ is the **regularization term** (maximizing the margin).
* $C$ is the **penalty parameter**.
* $\max(0, 1 - y_i(w \cdot x_i + b))$ is the **Hinge Loss**.

### Gradients of the Soft Margin SVM Loss Function

To minimize $L(w, b)$ using optimization techniques like Stochastic Gradient Descent (SGD), we compute the partial derivatives (subgradients):

#### 1. Gradient with respect to $w$:
$$\nabla_w L = w - C \sum_{i=1}^{n} \begin{cases} y_i x_i & \text{if } y_i(w \cdot x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

#### 2. Gradient with respect to $b$:
$$\nabla_b L = - C \sum_{i=1}^{n} \begin{cases} y_i & \text{if } y_i(w \cdot x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

In [23]:
class SVM:

    def __init__(self , epochs , lr , X , y , C):

        self.epochs = epochs
        self.lr = lr
        self.X = X
        self.y = y
        self.C = C
        self.w = np.zeros(X.shape[1])
        self.b = 0
    
    def train(self):

        for _ in range(self.epochs):
            n_samples = self.X.shape[0]

            for i in range(n_samples):

                output = self.y[i] * (np.dot(self.w , self.X[i]) + self.b)
                if(output >= 1):

                    dw = self.w
                    db = 0

                else:

                    dw = self.w - ( self.C * (self.y[i] * self.X[i]) )
                    db = -(self.C * self.y[i])
                
                self.w = self.w - (self.lr * dw)
                self.b = self.b - (self.lr * db)
    
    def predict(self , X_test):

        n_samples = X_test.shape[0]
        y_pred = []
        for i in range(n_samples):

            output = np.sign(np.dot(self.w , X_test[i]) + self.b)
            y_pred.append(output)

        return np.array(y_pred)

    def eval(self , y_pred , y_test):
        #accuracy
        return np.mean(y_pred == y_test)


        

In [24]:
from itertools import combinations
import numpy as np

class OneVOne:
    
    def __init__(self , lr=0.001, C=1.0, epochs=1000):

        self.lr = lr
        self.C = C
        self.epochs = epochs
        self.models = []

    def train(self , X_train , y_train):

        self.classes = np.unique(y_train)  
        self.class_to_index = {c:i for i,c in enumerate(self.classes)}

        pairs = combinations(self.classes , 2)

        for pair in pairs:
           
            idx = (y_train == pair[0]) | (y_train == pair[1])

            new_X = X_train[idx]
            new_y = y_train[idx]
            new_y = np.where(new_y == pair[0], -1, 1)

            svm = SVM(self.epochs , self.lr , new_X , new_y , self.C)

            svm.train()
            self.models.append((svm , pair[0] , pair[1]))

    def predict(self, X):

        votes = np.zeros((X.shape[0], len(self.classes)))

        for model, class1, class2 in self.models:

            pred = model.predict(X)

            for i, p in enumerate(pred):
                if p == -1:
                    votes[i, self.class_to_index[class1]] += 1
                else:
                    votes[i, self.class_to_index[class2]] += 1

        return self.classes[np.argmax(votes, axis=1)]


In [25]:
model = OneVOne(lr=0.001, C=1.0, epochs=1000)

model.train(X_train, y_train)


In [26]:
preds = model.predict(X_test)


In [28]:
accuracy = np.mean(preds == y_test)

print("Predictions:", preds[:10])
print("Actual:", y_test[:10])
print("Accuracy:", accuracy * 100)


Predictions: ['Iris-setosa' 'Iris-virginica' 'Iris-virginica' 'Iris-versicolor'
 'Iris-versicolor' 'Iris-setosa' 'Iris-versicolor' 'Iris-versicolor'
 'Iris-setosa' 'Iris-virginica']
Actual: 39         Iris-setosa
77     Iris-versicolor
116     Iris-virginica
73     Iris-versicolor
94     Iris-versicolor
26         Iris-setosa
60     Iris-versicolor
59     Iris-versicolor
27         Iris-setosa
129     Iris-virginica
Name: Species, dtype: str
Accuracy: 83.33333333333334
